In [ ]:
# ============================================================
# GitHub-renderelési hiba javítása:
# "the 'state' key is missing from 'metadata.widgets'"
#
# Ez a kód MEGTARTJA:
# - markdown cellákat
# - kódcellákat
# - print outputokat
# - táblázatos outputokat
# - képeket / ábrákat
#
# És TÖRLI:
# - metadata.widgets blokkot
# - Colab/Jupyter widget-hivatkozásokat
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# 1. EREDETI notebook elérési útja
# Ezt igazítsd ahhoz, ahol nálad a fájl van.
# ------------------------------------------------------------

input_path = Path("/content/drive/MyDrive/Privat/Study/PE/SZD/Kislanyjatek/SPSS_SAV_Colab_cells.ipynb")

# ------------------------------------------------------------
# 2. Új, GitHub-kompatibilis notebook neve
# ------------------------------------------------------------

output_path = Path("/content/drive/MyDrive/Privat/Study/PE/SZD/Kislanyjatek/SPSS_SAV_Colab_cells_GitHub_ready.ipynb")

# ------------------------------------------------------------
# 3. Notebook betöltése
# ------------------------------------------------------------

with open(input_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# ------------------------------------------------------------
# 4. A problémás notebook-szintű widgets blokk TELJES törlése
# ------------------------------------------------------------

if "metadata" in nb:
    nb["metadata"].pop("widgets", None)

# ------------------------------------------------------------
# 5. Cellaszintű Colab/Jupyter widget-metaadatok törlése
# ------------------------------------------------------------

for cell in nb.get("cells", []):
    metadata = cell.get("metadata", {})

    # Cellaszintű widget-hivatkozások törlése
    metadata.pop("widgets", None)
    metadata.pop("referenced_widgets", None)
    metadata.pop("outputId", None)

    # Colab-specifikus widget-hivatkozások törlése
    if "colab" in metadata and isinstance(metadata["colab"], dict):
        metadata["colab"].pop("referenced_widgets", None)

    # Outputokon belüli widget-specifikus MIME-típusok törlése
    for output in cell.get("outputs", []):
        data = output.get("data", {})
        if isinstance(data, dict):
            data.pop("application/vnd.jupyter.widget-view+json", None)
            data.pop("application/vnd.google.colaboratory.intrinsic+json", None)

# ------------------------------------------------------------
# 6. Execution count rendezése
# ------------------------------------------------------------

execution_counter = 1

for cell in nb.get("cells", []):
    if cell.get("cell_type") == "code":
        outputs = cell.get("outputs", [])

        if outputs:
            cell["execution_count"] = execution_counter

            for output in outputs:
                if "execution_count" in output:
                    output["execution_count"] = execution_counter

            execution_counter += 1
        else:
            cell["execution_count"] = None

# ------------------------------------------------------------
# 7. Ellenőrzés: maradt-e metadata.widgets?
# ------------------------------------------------------------

if "widgets" in nb.get("metadata", {}):
    raise ValueError("HIBA: még mindig maradt metadata.widgets blokk!")

# ------------------------------------------------------------
# 8. Mentés
# ------------------------------------------------------------

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Elkészült a javított GitHub-kompatibilis notebook:")
print(output_path)

Elkészült a javított GitHub-kompatibilis notebook:
/content/drive/MyDrive/Privat/Study/PE/SZD/Kislanyjatek/SPSS_SAV_Colab_cells_GitHub_ready.ipynb


In [ ]:
from google.colab import drive
drive.mount('/content/drive')